# AInstein component: LLM QA

In [1]:
import sys
from pathlib import Path

path_project = Path.cwd().parent
sys.path.append(str(path_project))

In [2]:
from AInstein import (
    get_llm_azure_openai, # conexión al modelo GPT
    get_settings, # conexión a las configuraciones de los modelos
    BigQueryManager
)

In [3]:
import re
import json
from collections import defaultdict

import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML
from openpyxl import load_workbook
from io import BytesIO
import copy

# Settings

In [5]:
# Settings
ENVIRONMENT: str = 'bdb-gcp-sbx-ia'
WORKPLACE_PROJECT_ID: str = 'geo-cargas_laborales'

# Obtener las configuraciones del proyecto
settings = get_settings(WORKPLACE_PROJECT_ID, environment=ENVIRONMENT)

# Models

In [6]:
# Crear instancias de los modelos
llm = get_llm_azure_openai(settings)  # Modelo de Azure OpenAI

# Responses

## General

In [77]:
class ProcesadorTranscripcionTeams:
    """
    Procesa archivos VTT de Microsoft Teams para levantar actividades,
    enriquecerlas con IA y generar resúmenes confirmables.
    """

    # ------------------------------------------------------------------
    # 1. LECTURA Y LIMPIEZA DEL VTT
    # ------------------------------------------------------------------
    def leer_archivo_vtt(self, contenido):
        conversaciones = []
        lineas = contenido.splitlines()

        timestamp_actual = None
        buffer_texto = []

        for linea in lineas:
            linea = linea.strip()

            if (
                not linea
                or linea == "WEBVTT"
                or re.match(r"^[a-f0-9\-]+\/\d+\-\d+$", linea)
            ):
                continue

            if "-->" in linea:
                if timestamp_actual and buffer_texto:
                    conversaciones.append({
                        "timestamp": timestamp_actual,
                        "texto": " ".join(buffer_texto).strip()
                    })
                    buffer_texto = []

                inicio = linea.split("-->")[0].strip()
                timestamp_actual = inicio.split(".")[0]
                continue

            linea = re.sub(r"<v[^>]*>", "", linea)
            linea = re.sub(r"</v>", "", linea)
            linea = re.sub(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ\s,]+:\s*", "", linea)

            if linea:
                buffer_texto.append(linea)

        if timestamp_actual and buffer_texto:
            conversaciones.append({
                "timestamp": timestamp_actual,
                "texto": " ".join(buffer_texto).strip()
            })

        return conversaciones

    # ------------------------------------------------------------------
    # 1.1 OBTENER CONTENIDO DEL EXCEL DE LAS ACTIVIDADES NO DIARIAS
    # ------------------------------------------------------------------
    def obtener_contenido_excel(self, upload_widget):
        if not upload_widget.value:
            raise ValueError("No se ha subido ningún archivo Excel")

        valor = upload_widget.value
        archivo = valor[0] if isinstance(valor, tuple) else list(valor.values())[0]
        contenido_raw = archivo["content"]

        if isinstance(contenido_raw, memoryview):
            return contenido_raw.tobytes()
        elif isinstance(contenido_raw, bytes):
            return contenido_raw
        else:
            raise TypeError("Tipo de contenido no soportado para Excel")
            
    # ------------------------------------------------------------------
    # 1.2 LEER EXCEL ACTIVIDADES NO DIARIAS
    # ------------------------------------------------------------------
    def leer_excel_actividades_no_diarias(self, contenido_excel):
        actividades = []

        wb = load_workbook(filename=BytesIO(contenido_excel), data_only=True)
        ws = wb.active

        encabezados = [str(c.value).strip() for c in ws[1]]

        idx_actividad = encabezados.index("DESCRIPCIÓN DE LA ACTIVIDAD")
        idx_frecuencia = encabezados.index("FRECUENCIA")
        idx_volumen = encabezados.index("VOL.(SEGÚN FRECUENCIA)")
        idx_duracionmin = encabezados.index("TIEMPO ESTIMADO POR UNIDAD EN MINUTOS")
        idx_procesoarea = encabezados.index("PROCESO DEL ÁREA")
        idx_autonomia = encabezados.index("AUTONOMÍA DE LA TAREA (0-100%)")
        idx_observaciones = encabezados.index("OBSERVACIONES")

        for fila in ws.iter_rows(min_row=2, values_only=True):
            if not fila[idx_actividad]:
                continue
                
            actividades.append({
                "nombre": fila[idx_actividad],
                "frecuencia": fila[idx_frecuencia].capitalize(),
                "volumen": fila[idx_volumen],
                "duracion_min": fila[idx_duracionmin],
                "proceso_area": fila[idx_procesoarea],
                "autonomia": round(fila[idx_autonomia]*100, 0),
                "observaciones": fila[idx_observaciones]
            })

        return actividades
            
    # ------------------------------------------------------------------
    # 2. EXTRACCIÓN DE ACTIVIDADES (SIN IA)
    # ------------------------------------------------------------------
    def extraer_actividades(self, conversaciones):
        actividades = []
        actividad_actual = None

        patrones_inicio = [
            r"\binicio\b",
            r"\biniciar\b",
            r"\bcomienzo\b",
            r"\bempiezo\b",
            r"\bvoy a iniciar\b"
        ]

        patrones_fin = [
            r"\bfinalizo\b",
            r"\btermino\b",
            r"\bterminar\b",
            r"\bfinalizar\b"
        ]

        for conv in conversaciones:
            texto = conv["texto"].lower()
            timestamp = conv["timestamp"]

            if any(re.search(p, texto) for p in patrones_inicio):
                actividad_actual = {
                    "descripcion": conv["texto"],
                    "inicio": timestamp,
                    "fin": None
                }

            elif actividad_actual and any(re.search(p, texto) for p in patrones_fin):
                actividad_actual["fin"] = timestamp
                actividades.append(actividad_actual)
                actividad_actual = None

        return actividades
    
    # ------------------------------------------------------------------
    # 3. CÁLCULO DE DURACIÓN
    # ------------------------------------------------------------------
    def calcular_duracion_minutos(self, inicio, fin):
        h1, m1, s1 = map(int, inicio.split(":"))
        h2, m2, s2 = map(int, fin.split(":"))

        t1 = h1 * 3600 + m1 * 60 + s1
        t2 = h2 * 3600 + m2 * 60 + s2

        segundos = t2 - t1
        return max(segundos // 60, 0)

    # ------------------------------------------------------------------
    # 4. PIPELINE DE PROCESAMIENTO
    # ------------------------------------------------------------------
    def procesar_archivo(self, contenido):
        conversaciones = self.leer_archivo_vtt(contenido)
        actividades = self.extraer_actividades(conversaciones)

        resultado = []
        for a in actividades:
            if not a["fin"]:
                continue

            duracion = self.calcular_duracion_minutos(a["inicio"], a["fin"])
            if duracion <= 0:
                continue

            resultado.append({
                "actividad": a["descripcion"],
                "inicio": a["inicio"],
                "fin": a["fin"],
                "duracion_min": duracion
            })

        return resultado

    # ------------------------------------------------------------------
    # 5. ENRIQUECIMIENTO CON IA (UNIDAD + PHVA)
    # ------------------------------------------------------------------
    def _parse_json_seguro(self, texto):
        texto = texto.strip()
    
        if not texto:
            raise ValueError("❌ El LLM devolvió una respuesta vacía")
    
        # Quitar markdown si aparece
        if texto.startswith("```"):
            texto = re.sub(r"```json|```", "", texto).strip()
    
        # Recortar hasta el primer [ o {
        inicio = min(
            [i for i in [texto.find("["), texto.find("{")] if i != -1],
            default=-1
        )
    
        if inicio > 0:
            texto = texto[inicio:]
    
        try:
            return json.loads(texto)
        except json.JSONDecodeError:
            print("❌ JSON inválido devuelto por el LLM")
            print("Respuesta cruda:")
            print(texto)
            raise

    def enriquecer_actividades(self, actividades, cargo, vicepresidencia):
    
        prompt = f"""
    Eres un analista experto en Levantamiento de Cargas Laborales.
    
    Contexto del colaborador:
    - Cargo: {cargo}
    - Vicepresidencia: {vicepresidencia}
    
    Para cada actividad, debes:
    - unidad_medida (ej: solicitudes, informes, reuniones, casos, desarrollos)
    - phva (Planear, Hacer, Verificar, Actuar)
    
    ⚠️ REGLAS ESTRICTAS:
    - Devuelve EXCLUSIVAMENTE un JSON válido
    - NO incluyas texto antes o después
    - NO expliques nada
    - NO uses markdown
    - Devuelve una LISTA del mismo tamaño que la entrada
    
    Formato exacto de salida:
    [
      {{
        "nombre": "texto corto",
        "unidad_medida": "texto",
        "phva": "Planear | Hacer | Verificar | Actuar"
      }}
    ]
    
    Actividades de entrada:
    {json.dumps(actividades, indent=2, ensure_ascii=False)}
    """
        reply = llm.invoke(prompt)
        return self._parse_json_seguro(reply.content)

    # ------------------------------------------------------------------
    # 6. CONSTRUCCIÓN DE RESUMEN ESTRUCTURADO
    # ------------------------------------------------------------------
    def construir_resumen_actividades(self, actividades_enriquecidas):
        resumen = ""

        for i, act in enumerate(actividades_enriquecidas, start=1):
            resumen += f"""
Actividad {i}:
- Descripción: {act['nombre']}
- Frecuencia: {act.get('frecuencia', 'No especificada')}
- Duración (minutos): {act.get('duracion_min')}
- Proceso del área: {act.get('proceso_area')}
- Volumen: {act.get('volumen', 'N/A')}
- Unidad de Medida: {act.get('unidad_medida')}
- Tipo de actividad (PHVA): {act.get('phva')}
- Autonomía: {act.get('autonomia', 'N/A')}%
"""
            if act.get("observaciones"):
                resumen += f"- Observaciones: {act['observaciones']}\n"

        return resumen

    # ------------------------------------------------------------------
    # 7. RESUMEN PARA CONFIRMACIÓN DEL USUARIO
    # ------------------------------------------------------------------
    def resumen_para_confirmacion(
        self,
        actividades_enriquecidas,
        contexto
    ):
        resumen_actividades = self.construir_resumen_actividades(
            actividades_enriquecidas
        )

        prompt = f"""
    Eres un asistente de Levantamiento de Cargas Laborales.
    
    Contexto del colaborador:
    - Cargo: {contexto['cargo']}
    - Vicepresidencia: {contexto['vicepresidencia']}
    
    A continuación se presenta un resumen de las actividades
    identificadas a partir de reuniones y la información ingresada
    por el colaborador.
    
    {resumen_actividades}
    
    Instrucciones:
    - Resume la información de forma clara y ordenada
    - No combines el Volumen con la Unidad de Medida, 
    dalos por separado ya que el volumen es respecto a la Frecuencia (ej: Volumen = 1 y Frecuencia = Diaria -> 1 vez al día)
    - Usa lenguaje sencillo y no técnico
    - No agregues información nueva
    - No hagas cálculos adicionales
    - No corrijas ortografía a menos que sean la misma palabra pero en mayusculas/minusculas/empezando por mayúsculas/empezando por minúsculas, es decir, 
    si se tiene por ejemplo Frecuencia: Quincenal en una actividad y en otra Frecuencia: Quincenal*, muestralo tal como está
    - Finaliza preguntando si la información es correcta o si desea hacer ajustes
    """
        reply = llm.invoke(prompt)
        return reply.content

    # ------------------------------------------------------------------
    # 8. LECTURA DE ARCHIVO DESDE FILEUPLOAD (JUPYTER)
    # ------------------------------------------------------------------
    def obtener_contenido_vtt(self, upload_widget):
        if not upload_widget.value:
            raise ValueError("No se ha subido ningún archivo")

        valor = upload_widget.value
        archivo = valor[0] if isinstance(valor, tuple) else list(valor.values())[0]
        contenido_raw = archivo["content"]

        if isinstance(contenido_raw, memoryview):
            return contenido_raw.tobytes().decode("utf-8")
        elif isinstance(contenido_raw, bytes):
            return contenido_raw.decode("utf-8")
        else:
            raise TypeError("Tipo de contenido no soportado")

     # ------------------------------------------------------------------
     # 8. LECTURA DE ARCHIVO DESDE FILEUPLOAD (JUPYTER)
     # ------------------------------------------------------------------
    def fusionar_inputs_usuario(self, actividades_base, actividades_enriquecidas, inputs_usuario):
        """
        Une actividades detectadas + enriquecidas por IA + inputs del usuario
        """
        actividades_finales = []
        
        for base, ia, user in zip(
            actividades_base,
            actividades_enriquecidas,
            inputs_usuario
        ):
            actividades_finales.append({
                "nombre": ia["nombre"],
                "duracion_min": base["duracion_min"],
        
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
        
                "frecuencia": user["frecuencia"],
                "volumen": user["volumen"],
                "proceso_area": user["proceso_area"],
                "autonomia": user["autonomia"],
                "observaciones": user.get("observaciones", "")
            })
        
        return actividades_finales

    # ------------------------------------------------------------------
     # 9. CONFIRMAR LA INFORMACIÓN POR PARTE DEL USUARIO
     # ------------------------------------------------------------------
    def confirmar_informacion(self, resumen_texto):
        print("📋 RESUMEN PARA CONFIRMACIÓN\n")
        print(resumen_texto)
        
        respuesta = input("\n¿La información es correcta? (si / no): ").strip().lower()
        return respuesta == "si"

    # ------------------------------------------------------------------
     # 10. CALCULAR LAS MÉTRICAS (falta poner el resto de cálculos)
     # -----------------------------------------------------------------
    def calcular_metricas(self, actividades):
        """
        Calcula métricas globales de carga laboral integrando:
        - tiempo
        - volumen
        - frecuencia
        - autonomía
        - PHVA
        - proceso / área
    
        Modelo alineado con estudio de cargas laborales.
        """
    
        # Constantes
        DIAS_LABORALES_MES = 21
        MINUTOS_JORNADA_MES = 220 * 60
        JORNADA_LABORAL_DIARIA = 8.5
    
        FACTOR_FRECUENCIA = {
            "Diario": 1,
            "Semanal": 1 / 5,
            "Quincenal": 1 / 10,
            "Mensual": 1 / 21,
            "Bimensual": 1 / 42,
            "Trimestral": 1 / 63,
            "Semestral": 1 / 126,
            "Anual": 1 / 252
        }
    
        # Acumuladores
        carga_w_por_phva = {}
        cantidad_por_phva = {}
    
        carga_w_por_frecuencia = {}
        cantidad_por_frecuencia = {}
    
        carga_w_por_proceso_area = {}
        cantidad_por_proceso_area = {}
    
        total_carga_w_sin_tm = 0
        total_carga_trabajo_individual = 0
        total_minutos_diarios = 0
    
        # Procesamiento Actividad
        for act in actividades:
            tiempo_base = act["duracion_min"]
            volumen = act["volumen"]
            frecuencia = act["frecuencia"]
            autonomia = act["autonomia"] / 100
    
            phva = act.get("phva", "SIN CLASIFICAR")
            proceso_area = act.get("proceso_area", "SIN CLASIFICAR")
    
            factor = FACTOR_FRECUENCIA.get(frecuencia, 0)
    
            minutos_diarios = tiempo_base * volumen * factor
            minutos_mes = minutos_diarios * DIAS_LABORALES_MES
    
            carga_w_sin_tm = minutos_mes / MINUTOS_JORNADA_MES
            carga_trabajo_individual = carga_w_sin_tm * autonomia
    
            # Guardar métricas por actividad (opcional pero MUY útil)
            act["metricas"] = {
                "minutos_diarios": round(minutos_diarios, 2),
                "minutos_mes": round(minutos_mes, 2),
                "carga_w_sin_tm": round(carga_w_sin_tm, 4),
                "carga_trabajo_individual": round(carga_trabajo_individual, 4)
            }
    
            total_carga_w_sin_tm += carga_w_sin_tm
            total_carga_trabajo_individual += carga_trabajo_individual
            total_minutos_diarios += minutos_diarios
    
            # ---- PHVA ----
            carga_w_por_phva[phva] = carga_w_por_phva.get(phva, 0) + carga_w_sin_tm
            cantidad_por_phva[phva] = cantidad_por_phva.get(phva, 0) + 1
    
            # ---- Frecuencia ----
            carga_w_por_frecuencia[frecuencia] = (
                carga_w_por_frecuencia.get(frecuencia, 0) + carga_w_sin_tm
            )
            cantidad_por_frecuencia[frecuencia] = (
                cantidad_por_frecuencia.get(frecuencia, 0) + 1
            )
    
            # ---- Proceso / Área ----
            carga_w_por_proceso_area[proceso_area] = (
                carga_w_por_proceso_area.get(proceso_area, 0) + carga_w_sin_tm
            )
            cantidad_por_proceso_area[proceso_area] = (
                cantidad_por_proceso_area.get(proceso_area, 0) + 1
            )
    
        # Tiempo Muerto
        almuerzo = 1 / 8
        baño_agua_etc = (3 * 5) / 60
        break_15min = 15 / 60
        latencia_software = 10 / 60
        cliente_ext_o_int = 20 / 60
    
        tiempo_muerto = sum([
            almuerzo / JORNADA_LABORAL_DIARIA,
            baño_agua_etc / JORNADA_LABORAL_DIARIA,
            break_15min / JORNADA_LABORAL_DIARIA,
            latencia_software / JORNADA_LABORAL_DIARIA,
            cliente_ext_o_int / JORNADA_LABORAL_DIARIA
        ])
    
        factor_tiempo_neto = round(1 - tiempo_muerto, 4)
    
        # Porcentajes
        def porcentajes(dic):
            total = sum(dic.values())
            return { 
                k: round(v / total, 4) if total > 0 else 0
                for k, v in dic.items()
            }
    
        # Resultado Final    
        horas_diarias_requeridas = total_minutos_diarios / 60
        horas_netas_por_persona = JORNADA_LABORAL_DIARIA * factor_tiempo_neto
    
        return {
            # ---- PHVA ----
            "carga_trabajo_phva": {k: round(v, 4) for k, v in carga_w_por_phva.items()},
            "cantidad_actividades_phva": cantidad_por_phva,
            "porcentaje_actividades_phva": porcentajes(carga_w_por_phva),
    
            # ---- Frecuencia ----
            "carga_trabajo_frecuencia": {k: round(v, 4) for k, v in carga_w_por_frecuencia.items()},
            "cantidad_actividades_frecuencia": cantidad_por_frecuencia,
            "porcentaje_actividades_frecuencia": porcentajes(carga_w_por_frecuencia),
    
            # ---- Proceso / Área ----
            "carga_trabajo_proceso_area": {
                k: round(v, 4) for k, v in carga_w_por_proceso_area.items()
            },
            "cantidad_actividades_proceso_area": cantidad_por_proceso_area,
            "porcentaje_actividades_proceso_area": porcentajes(carga_w_por_proceso_area),
    
            # ---- Totales ----
            "total_carga_w_sin_tm": round(total_carga_w_sin_tm, 4),
            "total_carga_trabajo_individual": round(total_carga_trabajo_individual, 4),
            "minutos_diarios_empleados": round(total_minutos_diarios, 2),
            "horas_diarias_requeridas": round(horas_diarias_requeridas, 2),
    
            # ---- Dotación ----
            "jornada_laboral_diaria": JORNADA_LABORAL_DIARIA,
            "factor_tiempo_neto_productivo": factor_tiempo_neto,
            "horas_netas_efectivas_por_persona": round(horas_netas_por_persona, 2),
            "numero_personas_requeridas": round((total_minutos_diarios * (1 + tiempo_muerto) / 60) / horas_netas_por_persona, 3),
    
            # ---- Ajuste final ----
            "tiempo_muerto": round(tiempo_muerto, 4),
            "horas_diarias_requeridas_final": round(
                total_minutos_diarios * (1 + tiempo_muerto) / 60, 2
            )
        }
    
    # ------------------------------------------------------------------
     # 11. ANÁLISIS DE LA IA PARA EL ANALISTA 
     # ------------------------------------------------------------------
    def analisis_analista_ia(self, actividades, metricas, contexto):
        prompt = f"""
    Eres un ANALISTA SENIOR en Levantamiento de Cargas Laborales y Dimensionamiento Operativo.
    Tu tarea NO es recalcular datos, sino INTERPRETAR, VALIDAR COHERENCIA y EMITIR JUICIO PROFESIONAL
    a partir de la información suministrada.
    
    =========================
    CONTEXTO ORGANIZACIONAL
    =========================
    Cargo: {contexto.get("cargo")}
    Vicepresidencia: {contexto.get("vicepresidencia")}
    
    =========================
    ACTIVIDADES ANALIZADAS
    =========================
    Cada actividad incluye duración, frecuencia, volumen, autonomía, PHVA y proceso/área.
    
    {json.dumps(actividades, indent=2, ensure_ascii=False)}
    
    =========================
    MÉTRICAS CALCULADAS
    =========================
    Las métricas ya incluyen:
    - Carga diaria y mensual
    - Distribución PHVA
    - Distribución por frecuencia
    - Autonomía promedio
    - Cálculo de dotación considerando tiempos muertos
    
    {json.dumps(metricas, indent=2, ensure_ascii=False)}
    
    =========================
    INSTRUCCIONES DE ANÁLISIS
    =========================
    
    1. Analiza el NIVEL DE CARGA LABORAL:
       - Usa las horas diarias requeridas finales y la dotación calculada.
       - Clasifica la carga como BAJA, ADECUADA o ALTA.
       - Justifica la clasificación con base en capacidad operativa real.
    
    2. Evalúa la COHERENCIA del resultado:
       - ¿Las métricas son consistentes con el tipo de cargo?
       - ¿Existen valores atípicos o concentraciones anómalas de carga?
       - ¿La autonomía declarada es coherente con la carga individual?
    
    3. Interpreta el BALANCE PHVA:
       - Explica qué indica la distribución PHVA sobre la naturaleza del rol.
       - Identifica posibles desviaciones (sobregestión, baja ejecución, ausencia de mejora).
    
    4. Identifica RIESGOS OPERATIVOS:
       - Riesgos de sobrecarga, dependencia, reprocesos o cuellos de botella.
       - Riesgos derivados de frecuencia, volumen o baja autonomía.
    
    5. Detecta OPORTUNIDADES DE AUTOMATIZACIÓN O MEJORA:
       - Actividades repetitivas, de alta frecuencia o bajo valor agregado.
       - Actividades con alta carga y baja autonomía.
    
    6. Formula RECOMENDACIONES EJECUTIVAS:
       - Orientadas a gestión del cargo, redistribución de carga o mejora operativa.
       - No propongas soluciones técnicas específicas, solo lineamientos.
    
    =========================
    FORMATO DE RESPUESTA (OBLIGATORIO)
    =========================
    
    Responde en español, con lenguaje profesional, estructurado en los siguientes bloques:
    
    1. Nivel de carga laboral (con justificación)
    2. Validación de coherencia de las métricas
    3. Interpretación del balance PHVA
    4. Riesgos operativos identificados
    5. Oportunidades de automatización o mejora
    6. Recomendaciones finales
    
    Quiero que menciones los valores numéricos mientras vas dando el análisis
    Interpreta y explica su significado.
    
    """
    
        reply = llm.invoke(prompt)
        
        return reply.content

    # ------------------------------------------------------------------
     # 12. CAPTURAR EL CARGO Y LA VICEPRESIDENCIA POR PARTE DEL USUARIO
     # ------------------------------------------------------------------
    def capturar_contexto_usuario(self):
        cargo = widgets.Text(
            description="Cargo:",
            placeholder="Ej: Analista de Datos",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
    
        vicepresidencia = widgets.Text(
            description="Vicepresidencia:",
            placeholder="Ej: Tecnología",
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
    
        display(cargo, vicepresidencia)
    
        return {
            "cargo": cargo,
            "vicepresidencia": vicepresidencia
        }

    # ------------------------------------------------------------------
     # 13. OBTENER EL CARGO Y LA VICEPRESIDENCIA
     # ------------------------------------------------------------------
    def obtener_contexto_valores(self, contexto_widgets):
        return {
            "cargo": contexto_widgets["cargo"].value,
            "vicepresidencia": contexto_widgets["vicepresidencia"].value
        }

    # ----------------------------------------------------------------------
     # 14. CAPTURAR EL RESTO DE INPUTS POR ACTIVIDAD POR PARTE DEL USUARIO
     # ---------------------------------------------------------------------
    def capturar_inputs_usuario(self):
        frecuencia = widgets.Dropdown(
            options=["Diario", "Mensual", "Semanal", "Quincenal", "Bimensual", "Trimestral", "Semestral", "Anual"],
            description="Frecuencia:",
            style={'description_width': '120px'}
        )
    
        volumen = widgets.IntText(
            description="Volumen:",
            value=1,
            style={'description_width': '120px'}
        )
    
        autonomia = widgets.IntSlider(
            description="Autonomía (%):",
            min=0,
            max=100,
            value=80,
            style={'description_width': '120px'}
        )

        proceso_area = widgets.Textarea(
            description="Proceso del área:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
    
        observaciones = widgets.Textarea(
            description="Observaciones:",
            layout=widgets.Layout(width='500px', height='80px'),
            style={'description_width': '120px'}
        )
    
        display(frecuencia, volumen, autonomia, proceso_area, observaciones)
    
        return {
            "frecuencia": frecuencia,
            "volumen": volumen,
            "proceso_area": proceso_area,
            "autonomia": autonomia,
            "observaciones": observaciones
        }

    def obtener_actividades_unicas(self, actividades):
        vistas = set()
        unicas = []

        for act in actividades:
            nombre = act["actividad"].strip().lower()
            if nombre not in vistas:
                vistas.add(nombre)
                unicas.append(act)
        return unicas

    def capturar_inputs_usuario_actividades_unicas(self, actividades):
        actividades_unicas = self.obtener_actividades_unicas(actividades)

        inputs_por_actividad = {}

        for act in actividades_unicas:
            print(f"\n Actividad: {act['actividad']}")
            inputs_por_actividad[act["actividad"].strip().lower()] = (
                self.capturar_inputs_usuario()
            )

        return inputs_por_actividad

    # ------------------------------------------------------------------
     # 15. OBTENER EL RESTO DE INPUTS
     # ------------------------------------------------------------------
    def obtener_inputs_usuario_valores(self, inputs_widgets):
        return {
            "frecuencia": inputs_widgets["frecuencia"].value,
            "volumen": inputs_widgets["volumen"].value,
            "autonomia": inputs_widgets["autonomia"].value,
            "proceso_area": inputs_widgets["proceso_area"].value,
            "observaciones": inputs_widgets["observaciones"].value
        }

    # ------------------------------------------------------------------
     # 16. LECTURA DE ARCHIVO DESDE FILEUPLOAD (JUPYTER)
     # ------------------------------------------------------------------
    def editar_actividades(self, actividades):
        """
        Permite al usuario modificar o eliminar actividades
        mostrando confirmación inmediata de cada cambio.
        """
    
        import copy
    
        while True:
            if not actividades:
                print("\n⚠️ No hay actividades para editar.")
                break
    
            print("\n✏️ ACTIVIDADES DISPONIBLES:")
            for i, act in enumerate(actividades, start=1):
                print(f"{i}. {act['nombre']}")
    
            opcion = input(
                "\nIngrese el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()
    
            if opcion == "salir":
                break
    
            if not opcion.isdigit() or not (1 <= int(opcion) <= len(actividades)):
                print("⚠️ Opción inválida")
                continue
    
            idx = int(opcion) - 1
            actividad = actividades[idx]
    
            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
            accion = input(
                "¿Qué deseas hacer? (editar / eliminar / cancelar): "
            ).strip().lower()
    
            if accion == "cancelar":
                continue
    
            # ==========================================================
            # ELIMINAR ACTIVIDAD
            # ==========================================================
            if accion == "eliminar":
    
                actividad_original = copy.deepcopy(actividad)
    
                confirmacion = input(
                    f"⚠️ ¿Seguro que deseas eliminar la actividad "
                    f"'{actividad['nombre']}'? (si / no): "
                ).strip().lower()
    
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
    
                    confirmar_final = input(
                        "¿Confirmas la eliminación? (si / no): "
                    ).strip().lower()
    
                    if confirmar_final != "si":
                        actividades.insert(idx, actividad_original)
                        print("↩️ Eliminación revertida.")
    
                continue
    
            # ==========================================================
            # EDITAR ACTIVIDAD
            # ==========================================================
            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue
    
            campos_editables = [
                "frecuencia",
                "volumen",
                "duracion_min",
                "autonomia",
                "observaciones",
                "unidad_medida",
                "phva"
            ]
    
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")
    
            campo = input("\nCampo a modificar: ").strip()
    
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue
    
            nuevo_valor = input("Nuevo valor: ").strip()
    
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)
    
            # 🔹 Guardar estado anterior
            actividad_original = copy.deepcopy(actividad)
    
            # 🔹 Aplicar cambio
            actividad[campo] = nuevo_valor
    
            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo modificado: {campo}")
            print(f"- Valor anterior: {actividad_original.get(campo)}")
            print(f"- Nuevo valor: {nuevo_valor}")
    
            confirmar_cambio = input(
                "\n¿Confirmas este cambio? (si / no): "
            ).strip().lower()
    
            if confirmar_cambio != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")
    
        return actividades


    # ------------------------------------------------------------------
    # 17. CAPTURA DE ACTIVIDADES NO DIARIAS (MANUAL)
    # ------------------------------------------------------------------
    def fusionar_inputs_usuario_nodiarias(self, actividades_base, actividades_enriquecidas):
        """
        Une actividades detectadas + enriquecidas por IA (no diarias)
        """
        actividades_finales_nodiarias = []
        
        for base, ia in zip(
            actividades_base,
            actividades_enriquecidas
        ):
            actividades_finales_nodiarias.append({
                "nombre": base["nombre"],
                "duracion_min": base["duracion_min"],
        
                "unidad_medida": ia["unidad_medida"],
                "phva": ia["phva"],
        
                "frecuencia": base["frecuencia"],
                "volumen": base["volumen"],
                "proceso_area": base["proceso_area"],
                "autonomia": base["autonomia"],
                "observaciones": base.get("observaciones", "")
            })
        
        return actividades_finales_nodiarias

    # ------------------------------------------------------------------
    # 18. VALIDACIÓN DE COHERENCIA GLOBAL (LLM SIN REGLAS)
    # ------------------------------------------------------------------
    def validar_coherencia_global(self, actividades, contexto):
        """
        Envía TODAS las actividades al modelo para que detecte
        posibles incoherencias lógicas o matemáticas.
    
        El modelo solo debe devolver actividades donde detecte
        incoherencias. Si no hay ninguna, debe indicarlo de forma explícita.
        """
    
        prompt = f"""
    Actúa como un validador técnico de coherencia de actividades laborales.
    
    Recibirás información estructurada sobre actividades con los siguientes campos:
    - nombre
    - frecuencia
    - volumen
    - duración por unidad
    - unidad de medida
    - otros campos opcionales
    
    Regla obligatoria:
    El volumen representa el número de veces que se ejecuta la actividad dentro del periodo definido por la frecuencia.
    
    Ejemplo:
    - Frecuencia: semanal + Volumen: 5 → 5 ejecuciones por semana.
    - Frecuencia: diaria + Volumen: 2 → 2 ejecuciones por día.
    - Frecuencia: quincenal + Volumen: 3 → 3 ejecuciones por quincena.
    
    Tu tarea:
    
    1. Analizar exclusivamente la información proporcionada.
    2. Detectar únicamente incoherencias internas.
    3. No hacer juicios del rol.
    4. No usar información externa.
    5. No inventar datos.
    6. No interpretar más allá de los valores dados.
    
    Debes validar únicamente:
    - volumen × duración dentro del periodo de la frecuencia
    - acumulaciones problemáticas dentro del mismo periodo
    - duraciones extremadamente bajas o altas en relación con el volumen
    - inconsistencias temporales si existen
    
    ⚠️ IMPORTANTE:
    
    - Solo debes mostrar las actividades donde detectes incoherencias.
    - NO debes mostrar actividades que estén correctas.
    - NO repitas la estructura para actividades sin problema.

    
    Formato de salida obligatorio:
    
    ------------------------------------------------------------
    
    🔎 VALIDACIÓN DE COHERENCIA
    
    Para cada actividad con incoherencia:
    
    📌 Actividad: [Nombre exacto]
    
    Campos evaluados:
    - Frecuencia: [valor]
    - Volumen: [valor]
    - Duración por unidad: [valor + unidad]
    
    Cálculo implícito:
    [volumen × duración = total dentro del periodo]
    
    Incoherencia detectada:
    - [Descripción técnica breve]
    
    ------------------------------------------------------------
    
    🧾 Resumen general:
    - Total de actividades analizadas: [número]
    - Actividades con incoherencias: [número]
    - Errores matemáticos explícitos: [sí/no]
    
    No agregues opiniones.
    No agregues recomendaciones.
    Sé técnico y conciso.

    ------------------------------------------------------------
    Además del reporte estructurado, al final debes devolver un bloque JSON 
    con el siguiente formato exacto:

    {{
        "actividades_incoherentes": ["Nombre 1", "Nombre 2"]
    }}

    Si no hay incoherencias:

    {{
        "actividades_incoherentes": []
    }}
    
    
    =========================
    CONTEXTO
    =========================
    Cargo: {contexto.get("cargo")}
    Vicepresidencia: {contexto.get("vicepresidencia")}
    
    =========================
    ACTIVIDADES REGISTRADAS
    =========================
    {json.dumps(actividades, indent=2, ensure_ascii=False)}
    
    Responde en español.
    """
    
        reply = llm.invoke(prompt)
        content = reply.content
        
        actividades_incoherentes = []
        
        try:
            # Buscar el último bloque JSON del texto
            json_start = content.rfind("{")
            if json_start != -1:
                json_text = content[json_start:]
                json_data = json.loads(json_text)
                actividades_incoherentes = json_data.get(
                    "actividades_incoherentes",
                    []
                )
        except Exception:
            actividades_incoherentes = []
        
        # 🔹 Ahora sí devolvemos ambos valores
        return content, actividades_incoherentes
    # ------------------------------------------------------------------
    # 18.1 EDITAR ACTIVIDADES INCOHERENTES
    # ------------------------------------------------------------------
    def editar_actividades_incoherentes(self, actividades, actividades_incoherentes):
        # 🔒 Filtrar solo actividades incoherentes
        actividades_filtradas = [
            act for act in actividades
            if act["nombre"] in actividades_incoherentes
        ]
    
        if not actividades_filtradas:
            print("\n✅ No hay actividades con incoherencias para editar.")
            return actividades
    
        while True:
    
            print("\n⚠️ ACTIVIDADES CON INCOHERENCIAS:")
            for i, act in enumerate(actividades_filtradas, start=1):
                print(f"{i}. {act['nombre']}")
    
            opcion = input(
                "\nSeleccione el número de la actividad "
                "(o 'salir' para terminar ajustes): "
            ).strip().lower()
    
            if opcion == "salir":
                break
    
            if not opcion.isdigit() or not (1 <= int(opcion) <= len(actividades_filtradas)):
                print("⚠️ Opción inválida")
                continue
    
            actividad = actividades_filtradas[int(opcion) - 1]
    
            # 🔎 Buscar índice real en lista original
            idx = next(i for i, a in enumerate(actividades)
                       if a["nombre"] == actividad["nombre"])
    
            print(f"\n🔧 Actividad seleccionada: {actividad['nombre']}")
    
            accion = input("¿Qué deseas hacer? (editar / eliminar / cancelar): ").strip().lower()
    
            if accion == "cancelar":
                continue
    
            if accion == "eliminar":
                confirmacion = input(
                    f"¿Confirmas eliminar '{actividad['nombre']}'? (si / no): "
                ).strip().lower()
    
                if confirmacion == "si":
                    actividades.pop(idx)
                    print("🗑️ Actividad eliminada.")
                continue
    
            if accion != "editar":
                print("⚠️ Acción no válida.")
                continue
    
            campos_editables = [
                "frecuencia",
                "volumen",
                "duracion_min",
                "autonomia",
                "observaciones",
                "unidad_medida",
                "phva"
            ]
    
            print("\nCampos editables:")
            for c in campos_editables:
                print(f"- {c} (actual: {actividad.get(c)})")
    
            campo = input("\nCampo a modificar: ").strip()
    
            if campo not in campos_editables:
                print("⚠️ Campo no editable")
                continue
    
            nuevo_valor = input("Nuevo valor: ").strip()
    
            if campo in ("volumen", "duracion_min", "autonomia"):
                nuevo_valor = int(nuevo_valor)
    
            actividad_original = copy.deepcopy(actividad)
            actividades[idx][campo] = nuevo_valor
    
            print("\n🧾 RESUMEN DEL CAMBIO:")
            print(f"- Actividad: {actividad['nombre']}")
            print(f"- Campo: {campo}")
            print(f"- Antes: {actividad_original.get(campo)}")
            print(f"- Después: {nuevo_valor}")
    
            confirmar = input("\n¿Confirmas el cambio? (si / no): ").strip().lower()
    
            if confirmar != "si":
                actividades[idx] = actividad_original
                print("↩️ Cambio revertido.")
            else:
                print("✅ Cambio confirmado.")
    
        return actividades


    # ------------------------------------------------------------------
    # 19. RESUMEN AUTOMÁTICO DE CAMBIOS
    # ------------------------------------------------------------------
    def generar_resumen_cambios(self, antes, despues):
        """
        Compara dos versiones de actividades y genera
        un resumen claro de los cambios realizados.
        """
    
        prompt = f"""
    Eres un asistente que compara versiones de información.
    
    Tu tarea es identificar y resumir los cambios entre dos versiones
    de actividades laborales.
    
    REGLAS:
    - No recalcules métricas.
    - No hagas análisis.
    - Solo describe qué cambió.
    - Sé breve y claro.
    - Si no hubo cambios, indícalo.
    
    ====================
    ANTES
    ====================
    {json.dumps(antes, indent=2, ensure_ascii=False)}
    
    ====================
    DESPUÉS
    ====================
    {json.dumps(despues, indent=2, ensure_ascii=False)}
    
    Responde en español en formato claro y organizado.
    """
    
        reply = llm.invoke(prompt)
        return reply.content

#### Parte del Usuario

In [78]:
def main():
    print("\n📋 ANALIZADOR DE ACTIVIDADES DESDE TEAMS")
    
    print("\n Suba la transcripción de Teams (.vtt) - actividades diarias")
    upload_vtt = widgets.FileUpload(
        accept=".vtt",
        multiple=False
    )

    print("\n Suba el Excel con las actividades NO diarias")
    upload_excel = widgets.FileUpload(
        accept=".xlsx",
        multiple=False
    )
    
    display(upload_vtt, upload_excel)
    return upload_vtt, upload_excel

In [10]:
upload_vtt, upload_excel = main()


📋 ANALIZADOR DE ACTIVIDADES DESDE TEAMS

 Suba la transcripción de Teams (.vtt) - actividades diarias

 Suba el Excel con las actividades NO diarias


FileUpload(value=(), accept='.vtt', description='Upload')

FileUpload(value=(), accept='.xlsx', description='Upload')

In [79]:
procesador = ProcesadorTranscripcionTeams()

In [80]:
contexto_widgets = procesador.capturar_contexto_usuario()

Text(value='', description='Cargo:', layout=Layout(width='400px'), placeholder='Ej: Analista de Datos', style=…

Text(value='', description='Vicepresidencia:', layout=Layout(width='400px'), placeholder='Ej: Tecnología', sty…

In [81]:
contexto = procesador.obtener_contexto_valores(contexto_widgets)

In [82]:
contenido_vtt = procesador.obtener_contenido_vtt(upload_vtt)
actividades_base_diarias = procesador.procesar_archivo(contenido_vtt)

In [83]:
contenido_excel = procesador.obtener_contenido_excel(upload_excel)
actividades_base_nodiarias = procesador.leer_excel_actividades_no_diarias(contenido_excel)

In [84]:
inputs_usuario_widgets = []

actividades_unicas = procesador.obtener_actividades_unicas(actividades_base_diarias)

for actividad in actividades_unicas:
    print(f"\n📝 Actividad: {actividad['actividad']}")
    
    widgets_act = procesador.capturar_inputs_usuario()
    inputs_usuario_widgets.append(widgets_act)


📝 Actividad: Voy a iniciar la actividad de Revision de plan de trabajo individual y envio correo.


Dropdown(description='Frecuencia:', options=('Diario', 'Mensual', 'Semanal', 'Quincenal', 'Bimensual', 'Trimes…

IntText(value=1, description='Volumen:', style=DescriptionStyle(description_width='120px'))

IntSlider(value=80, description='Autonomía (%):', style=SliderStyle(description_width='120px'))

Textarea(value='', description='Proceso del área:', layout=Layout(height='80px', width='500px'), style=TextSty…

Textarea(value='', description='Observaciones:', layout=Layout(height='80px', width='500px'), style=TextStyle(…

In [85]:
inputs_usuario = []

for widgets_act in inputs_usuario_widgets:
    valores = procesador.obtener_inputs_usuario_valores(widgets_act)
    inputs_usuario.append(valores)

In [86]:
actividades_ia_diarias = procesador.enriquecer_actividades(actividades_base_diarias, cargo=contexto["cargo"], vicepresidencia=contexto["vicepresidencia"])
actividades_ia_nodiarias = procesador.enriquecer_actividades(actividades_base_nodiarias, cargo=contexto["cargo"], vicepresidencia=contexto["vicepresidencia"])

In [87]:
actividades_finales_diarias = procesador.fusionar_inputs_usuario(
    actividades_base_diarias,
    actividades_ia_diarias,
    inputs_usuario
)

In [88]:
actividades_finales_nodiarias = procesador.fusionar_inputs_usuario_nodiarias(
    actividades_base_nodiarias,
    actividades_ia_nodiarias
)

In [89]:
actividades_finales_final = actividades_finales_diarias + actividades_finales_nodiarias

In [91]:
# 🔎 Validación automática
mensaje_validacion, actividades_incoherentes = procesador.validar_coherencia_global(
    actividades_finales_final,
    contexto
)

print("\n🔎 VALIDACIÓN AUTOMÁTICA DE COHERENCIA\n")
print(mensaje_validacion)

# Si no hay incoherencias, no permitir edición
if not actividades_incoherentes:
    print("\n✅ No hay actividades con incoherencias. No se requieren ajustes.")
    actividades_totales = actividades_finales_final

else:
    confirmar = input(
        "\n¿Desea corregir las actividades con incoherencias? (si / no): "
    ).strip().lower()

    if confirmar == "si":

        # 🔹 Guardar versión anterior
        actividades_antes = copy.deepcopy(actividades_finales_final)

        # 🔹 Solo permite editar incoherentes
        actividades_despues = procesador.editar_actividades_incoherentes(
            actividades_finales_final,
            actividades_incoherentes
        )

        # 🔹 Generar resumen de cambios
        resumen = procesador.generar_resumen_cambios(
            actividades_antes,
            actividades_despues
        )

        print("\n🧾 RESUMEN DE CAMBIOS REALIZADOS\n")
        print(resumen)

        confirmar_final = input(
            "\n¿Confirma que estos cambios son correctos? (si / no): "
        ).strip().lower()

        if confirmar_final != "si":
            print("Puede volver a editar la información.")
            # Aquí podrías repetir el ciclo si quieres

        actividades_totales = actividades_despues

    else:
        actividades_totales = actividades_finales_final


🔎 VALIDACIÓN AUTOMÁTICA DE COHERENCIA

------------------------------------------------------------

🔎 VALIDACIÓN DE COHERENCIA

📌 Actividad: Revisión de plan de trabajo individual y envío de correo

Campos evaluados:
- Frecuencia: Diario
- Volumen: 3
- Duración por unidad: 5 min

Cálculo implícito:
3 × 5 min = 15 min diarios

Incoherencia detectada:
- Duración total diaria baja para 3 ejecuciones diarias, podría ser inconsistente con la naturaleza de la actividad (duración extremadamente baja en relación con el volumen).

📌 Actividad: steerco

Campos evaluados:
- Frecuencia: Trimestral
- Volumen: 2
- Duración por unidad: 120 min

Cálculo implícito:
2 × 120 min = 240 min (4 horas) trimestrales

Incoherencia detectada:
- Volumen alto para frecuencia trimestral, 2 reuniones de 2 horas cada una en un trimestre puede ser excesivo o poco común (acumulación problemática dentro del periodo).

------------------------------------------------------------

🧾 Resumen general:
- Total de activida


¿Desea corregir las actividades con incoherencias? (si / no):  si



⚠️ ACTIVIDADES CON INCOHERENCIAS:
1. Revisión de plan de trabajo individual y envío de correo
2. steerco



Seleccione el número de la actividad (o 'salir' para terminar ajustes):  1



🔧 Actividad seleccionada: Revisión de plan de trabajo individual y envío de correo


...


¿Confirmas el cambio? (si / no):  si


✅ Cambio confirmado.

⚠️ ACTIVIDADES CON INCOHERENCIAS:
1. Revisión de plan de trabajo individual y envío de correo
2. steerco



Seleccione el número de la actividad (o 'salir' para terminar ajustes):  salir



🧾 RESUMEN DE CAMBIOS REALIZADOS

Cambios identificados entre las versiones:

- En la actividad "Revisión de plan de trabajo individual y envío de correo":
  - El volumen cambió de 3 a 2 informes.

No se registraron otros cambios en las actividades restantes.



¿Confirma que estos cambios son correctos? (si / no):  si


In [40]:
resumen = procesador.resumen_para_confirmacion(
    actividades_finales_final,
    contexto
)

In [41]:
if procesador.confirmar_informacion(resumen):
    metricas = procesador.calcular_metricas(actividades_finales_final)
    analisis = procesador.analisis_analista_ia(
        actividades_finales_final,
        metricas,
        contexto
    )
else:
    print("✏️ El usuario solicitó ajustes.")
    procesador.editar_actividades(actividades_finales_final)

📋 RESUMEN PARA CONFIRMACIÓN

Resumen de actividades del colaborador (Gerente, Vicepresidencia Talento y Administrativa):

1. Revisión de plan de trabajo individual y envío de correo  
   - Frecuencia: Diario  
   - Duración: 30 minutos  
   - Volumen: 3 informes por día  
   - Proceso: Desarrollo institucional de nuevas formas de trabajo y agilidad  
   - Tipo: Verificar  
   - Autonomía: 100%  
   - Observaciones: Revisión de pendientes y nuevas solicitudes  

2. Liderar diseño y actualización del roadmap de Delivery en Negocio  
   - Frecuencia: Semanal  
   - Duración: 60 minutos  
   - Volumen: 2 reuniones por semana  
   - Proceso: Consolidación del equipo de Cultura y Transformación  
   - Tipo: Planear  
   - Autonomía: 100%  
   - Observaciones: Sesión de alineación de prioridades con el equipo de leads de agilidad  

3. Liderar diseño y actualización del roadmap de transformación organizacional  
   - Frecuencia: Semanal  
   - Duración: 120 minutos  
   - Volumen: 1 reunión p


¿La información es correcta? (si / no):  si


In [133]:
# Carga a BigQuery



#### Parte del Analista

In [179]:
resumen = procesador.resumen_para_confirmacion(
    actividades_finales_nodiarias,
    contexto
)

In [180]:
metricas = procesador.calcular_metricas(actividades_finales_final)

In [181]:
analisis = procesador.analisis_analista_ia(
        actividades_finales_final,
        metricas,
        contexto
    )

In [42]:
display(Markdown("## 📋 Actividades"))

df_act = pd.DataFrame(actividades_finales_final)
display(df_act)

display(Markdown("## 📊 Métricas por PHVA"))
df_phva = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_phva"],
    "# Act.": metricas["cantidad_actividades_phva"],
    "% Act.": metricas["porcentaje_actividades_phva"]
})
display(df_phva)

display(Markdown("## 📊 Métricas por FRECUENCIA"))
df_frecuencia = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_frecuencia"],
    "# Act.": metricas["cantidad_actividades_frecuencia"],
    "% Act.": metricas["porcentaje_actividades_frecuencia"]
})
display(df_frecuencia)

display(Markdown("## 📊 Métricas por PROCESO"))
df_proceso = pd.DataFrame({
    "Carga de Trabajo": metricas["carga_trabajo_proceso_area"],
    "# Act.": metricas["cantidad_actividades_proceso_area"],
    "% Act.": metricas["porcentaje_actividades_proceso_area"]
})
display(df_proceso)

display(Markdown("## 📊 TOTALES"))
df_totales = pd.DataFrame([{
    "Total Carga W sin TM": metricas["total_carga_w_sin_tm"],
    "Total Carga Trabajo Individual": metricas["total_carga_trabajo_individual"],
    "Minutos diarios empleados": metricas["minutos_diarios_empleados"],
    "Horas diarias requeridas": metricas["horas_diarias_requeridas"]
}])
display(df_totales)

display(Markdown("## 📊 DOTACIÓN"))
df_dotacion = pd.DataFrame([{
    "Jornada Laboral Diaria": metricas["jornada_laboral_diaria"],
    "Factor Tiempo Neto Productivo": metricas["factor_tiempo_neto_productivo"],
    "Horas Netas Efectivas por Persona": metricas["horas_netas_efectivas_por_persona"],
    "Número Personas Requeridas": metricas["numero_personas_requeridas"]
}])
display(df_dotacion)

display(Markdown("## 📊 AJUSTE FINAL"))
df_ajuste = pd.DataFrame([{
    "Tiempo Muerto": metricas["tiempo_muerto"],
    "Horas Diarias Requeridas Final": metricas["horas_diarias_requeridas_final"]
}])
display(df_ajuste)

display(Markdown("## 🧠 Análisis del Analista IA"))
display(Markdown(analisis))

## 📋 Actividades

,nombre,duracion_min,unidad_medida,phva,frecuencia,volumen,proceso_area,autonomia,observaciones,metricas
0,Revisión de plan de trabajo individual y envío...,30,informes,Verificar,Diario,3,Desarrollo institucional de nuevas formas de t...,100.0,Revision de pendientes y nuevas solicitudes.,"{'minutos_diarios': 90, 'minutos_mes': 1890, '..."
1,Liderar el diseño y actualización del roadmap...,60,reuniones,Planear,Semanal,2,Consolidación del equipo de Cultura y Transfor...,100.0,Sesion de alineación de prioridades con el equ...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
2,Liderar el diseño y actualización del roadmap...,120,reuniones,Planear,Semanal,1,Consolidación del equipo de Cultura y Transfor...,65.0,Alineación Lideres Dirección Cultura y Transfo...,"{'minutos_diarios': 24.0, 'minutos_mes': 504.0..."
3,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
4,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
5,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
6,Revisión del roadmap de delivery con el equipo...,60,reuniones,Verificar,Quincenal,1,Desarrollo institucional de nuevas formas de t...,30.0,Revision de avances en plan de trabajo de agil...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
7,Sesion de alineación de agilidad con aval (adl),60,reuniones,Planear,Quincenal,1,Desarrollo institucional de nuevas formas de t...,20.0,Alineación de formas de trabajo con areas de a...,"{'minutos_diarios': 6.0, 'minutos_mes': 126.0,..."
8,Agile GO Frente personas,90,seguimientos,Hacer,Semanal,1,Desarrollo institucional de nuevas formas de t...,10.0,Seguimiento a la ejecución de la estrategia co...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."
9,Agile GO Frente empresas,90,seguimientos,Hacer,Semanal,1,Desarrollo institucional de nuevas formas de t...,10.0,Seguimiento a la ejecución de la estrategia c...,"{'minutos_diarios': 18.0, 'minutos_mes': 378.0..."


## 📊 Métricas por PHVA

,Carga de Trabajo,# Act.,% Act.
Verificar,0.1874,6,0.3252
Planear,0.1744,7,0.3026
Hacer,0.2100,4,0.3644
Actuar,0.0045,1,0.0079


## 📊 Métricas por FRECUENCIA

...

,Jornada Laboral Diaria,Factor Tiempo Neto Productivo,Horas Netas Efectivas por Persona,Número Personas Requeridas
0,8.5,0.8676,7.37,0.927


## 📊 AJUSTE FINAL

,Tiempo Muerto,Horas Diarias Requeridas Final
0,0.1324,6.84


## 🧠 Análisis del Analista IA

1. **Nivel de carga laboral (con justificación)**

La carga laboral total calculada para el cargo de Gerente en la Vicepresidencia de Talento y Administrativa es de aproximadamente **6.84 horas diarias requeridas finales**, sobre una jornada laboral estándar de **8.5 horas diarias**. Esto representa un nivel de ocupación del **80.5%** de la jornada laboral disponible (6.84/8.5). La dotación requerida es de **0.927 personas**, lo que indica que la carga está dimensionada para una sola persona, sin necesidad de refuerzos adicionales.

Este nivel de carga puede clasificarse como **ADECUADO**. La carga no es excesiva ni baja; permite un margen para imprevistos, pausas y actividades no planificadas, considerando además un factor de tiempo neto productivo del **86.76%**. La carga diaria en minutos es de **362.29 minutos** (6.04 horas netas antes de ajustar tiempos muertos), lo que es razonable para un cargo gerencial con múltiples responsabilidades estratégicas y operativas.

2. **Validación de coherencia de las métricas**

Las métricas presentadas son coherentes con un cargo gerencial de alta responsabilidad y autonomía. La autonomía promedio declarada es alta, con valores que oscilan entre el **10% y 100%**, predominando actividades con autonomía superior al 50%, lo cual es consistente con la naturaleza del cargo que implica toma de decisiones y liderazgo.

No se observan valores atípicos en duración o volumen que distorsionen la carga total. La distribución de frecuencias muestra predominancia de actividades semanales (62.93% de las actividades y 36.27% de la carga), lo cual es típico en roles gerenciales que requieren seguimiento constante pero no microgestión diaria excesiva. La carga diaria (14.32%) corresponde a una sola actividad de revisión de plan de trabajo, lo que es razonable.

Se detecta una concentración significativa de carga en el proceso "Desarrollo institucional de nuevas formas de trabajo y agilidad" (61.91% de las actividades y 35.68% de la carga), lo que es coherente con la función estratégica del cargo, aunque se debe vigilar que no genere dependencia excesiva en un solo ámbito.

3. **Interpretación del balance PHVA**

La distribución de carga por PHVA es:

- Hacer: 21.0%
- Verificar: 18.74%
- Planear: 17.44%
- Actuar: 0.45%

El rol presenta un balance relativamente equilibrado entre las fases de **Hacer**, **Verificar** y **Planear**, con una ligera preponderancia en la ejecución (Hacer) y verificación. Esto indica que el gerente no solo diseña y planifica estrategias, sino que también participa activamente en el seguimiento y control de las iniciativas.

La fase de **Actuar** está subrepresentada (0.45%), lo que puede reflejar que las acciones correctivas o de mejora continua se delegan o se integran dentro de otras fases. No se observa una sobregestión ni una baja ejecución, pero la baja carga en Actuar podría ser un área a fortalecer para asegurar la mejora continua.

4. **Riesgos operativos identificados**

- **Sobrecarga puntual en actividades de alta frecuencia**: La actividad diaria de revisión de planes y correos consume un 14.32% de la carga, lo que puede generar fatiga si se suma a otras responsabilidades urgentes.
- **Dependencia en el proceso de agilidad**: Más del 60% de las actividades están concentradas en "Desarrollo institucional de nuevas formas de trabajo y agilidad", lo que puede generar cuellos de botella si el gerente no puede atender todas las demandas.
- **Baja autonomía en algunas actividades críticas**: Actividades como "Agile GO Frente personas" y "Sesión de alineación con aval (adl)" tienen autonomías bajas (10% y 20%), lo que puede generar dependencia de otros actores y retrasos.
- **Frecuencias quincenales y trimestrales con baja carga, pero posibles picos de trabajo**: Actividades como steerco y feria Bdb, aunque poco frecuentes, requieren dedicación concentrada que puede afectar la planificación semanal.

5. **Oportunidades de automatización o mejora**

- La actividad diaria de revisión de planes y envío de correos (90 minutos diarios) es repetitiva y con alta frecuencia; podría beneficiarse de herramientas que automaticen alertas o resúmenes para optimizar tiempo.
- Actividades con baja autonomía y alta duración, como las sesiones de alineación y seguimiento (60-90 minutos), podrían mejorar su eficiencia mediante agendas más estructuradas o predefinidas.
- La gestión de iniciativas estratégicas transversales (5 casos semanales, 60 minutos diarios) representa una carga significativa (9.55%) y es 100% autónoma; sin embargo, puede ser un área para estandarizar procesos o delegar parcialmente.
- La formación continua semanal (120 minutos) es necesaria, pero podría optimizarse con formatos flexibles o integrados en la rutina.

6. **Recomendaciones finales**

- Mantener el nivel de carga actual, dado que es adecuado, pero vigilar la concentración en el proceso de agilidad para evitar dependencia excesiva y posibles cuellos de botella.
- Fomentar un mayor énfasis en la fase de Actuar para fortalecer la mejora continua y evitar que las acciones correctivas se diluyan o deleguen sin seguimiento.
- Evaluar la redistribución o delegación de actividades con baja autonomía y alta duración para mejorar la eficiencia operativa y reducir riesgos de dependencia.
- Promover la implementación de mecanismos de automatización o estandarización en actividades de alta frecuencia y repetitividad, especialmente en la revisión diaria de planes y gestión de iniciativas.
- Considerar la flexibilización o integración de la formación continua para optimizar el tiempo sin afectar la actualización profesional.
- Finalmente, asegurar que el gerente cuente con espacios para actividades no planificadas o estratégicas que no están reflejadas en la carga actual, para mantener la capacidad de respuesta y liderazgo efectivo.

In [43]:
dfs = {
    "Actividades": df_act,
    "PHVA": df_phva,
    "Dotacion": df_dotacion,
    "Ajuste": df_ajuste,
    "Frecuencia": df_frecuencia,
    "Totales": df_totales,
    "Proceso_Area": df_proceso
}

In [44]:
ruta_excel = f"analisis_carga_laboral_{contexto['cargo']}.xlsx"

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    df_act.to_excel(writer, sheet_name="Actividades", index=True)
    df_phva.to_excel(writer, sheet_name="Métricas PHVA", index=True)
    df_frecuencia.to_excel(writer, sheet_name="Frecuencias", index=True)
    df_proceso.to_excel(writer, sheet_name="Procesos", index=True)
    df_dotacion.to_excel(writer, sheet_name="Dotacion", index=False)
    df_ajuste.to_excel(writer, sheet_name="Ajuste", index=False)
    df_totales.to_excel(writer, sheet_name="Totales", index=False)
    
    lineas_analisis = analisis.split("\n")

    df_analisis = pd.DataFrame({
        "Análisis del Analista IA": lineas_analisis
    })

    df_analisis.to_excel(writer, sheet_name='Analisis IA', index=False)

In [45]:
wb = load_workbook(ruta_excel)

for sheet in wb.sheetnames:
    ws = wb[sheet]
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = 25

wb.save(ruta_excel)

In [46]:
display(HTML(f"""
<a href="{ruta_excel}" download>
📥 Descargar archivo Excel de análisis de carga laboral
</a>
"""))

# Guardado en BigQuery